# mssql_python — praktyczny przewodnik: połączenie, odczyt do DataFrame, operacje DML

Notebook krok po kroku dla oficjalnego, nowego (GA) sterownika Microsoftu
`mssql-python` (pakiet PyPI: `mssql-python`, moduł do importu: `mssql_python`) —
następcy `pyodbc` w wielu nowych projektach, bo nie wymaga osobnej instalacji
sterownika ODBC (bundluje DDBC — Direct Database Connectivity).

**Instalacja:**
```
pip install mssql-python
```

In [3]:
from pathlib import Path

import pandas as pd
import polars as pl
import duckdb
import mssql_python

## 1. Połączenie z bazą danych

### Connection string — co jest wymagane, a co nie

| Element | Wymagany? | Komentarz |
|---|---|---|
| `Server` | **tak** | nazwa serwera, opcjonalnie `,port` (domyślnie 1433) |
| `Database` | **tak** (w praktyce) | bez tego łączysz się do bazy domyślnej użytkownika |
| `UID` / `PWD` | tak — **tylko przy autentykacji SQL** | pomiń przy `Trusted_Connection=yes` (Windows) lub `Authentication=ActiveDirectory...` (Azure/Entra) |
| `Encrypt` | nie, ale zalecane `yes` | wymagane domyślnie w nowszych wersjach sterownika |
| `TrustServerCertificate` | nie | ustaw `yes` tylko lokalnie z certyfikatem self-signed; w produkcji zostaw `no` |
| `Trusted_Connection` | nie | `yes` = użyj zalogowanego konta Windows zamiast UID/PWD |
| `Authentication` | nie | np. `ActiveDirectoryDefault`/`ActiveDirectoryInteractive` — do Azure SQL bez hasła w kodzie |

Connection string trzymaj w zmiennych środowiskowych / pliku `.env`
(`python-dotenv`) — nigdy na sztywno w repo.

In [ ]:
# --- SQL Server lokalny / on-premise, autentykacja SQL (typowy przypadek w pracy) ---
# CONNECTION_STRING = (
#     "Server=localhost,1433;"
#     "Database=MojaBaza;"
#     "UID=uzytkownik;"
#     "PWD=haslo;"
#     "Encrypt=yes;"
#     "TrustServerCertificate=yes"   # tylko na potrzeby lokalnego dev
# )

# --- Azure SQL / Fabric, bez hasła w kodzie (Entra ID) ---
# CONNECTION_STRING = (
#     "Server=twoj_serwer.database.windows.net;"
#     "Database=MojaBaza;"
#     "Encrypt=yes;"
#     "Authentication=ActiveDirectoryDefault"
# )

# --- z .env zamiast wpisywania na sztywno ---
# from dotenv import load_dotenv
# from os import getenv
# load_dotenv()
# CONNECTION_STRING = getenv("SQL_CONNECTION_STRING")

# conn = connect(CONNECTION_STRING)
# cursor = conn.cursor()

### Logowanie do bazy — zalecany proces (zmienne środowiskowe, nie na sztywno w kodzie)

Standardowy przepływ w projekcie/repo, krok po kroku:

1. **Utwórz plik `.env`** w katalogu projektu (obok notebooka/skryptu) z
   parametrami połączenia — nigdy nie wpisuj ich bezpośrednio w kodzie.
2. **Dodaj `.env` do `.gitignore`** — plik nie może trafić do repozytorium.
3. **Zainstaluj `python-dotenv`**: `pip install python-dotenv`.
4. **Wczytaj zmienne** na starcie notebooka/skryptu przez `load_dotenv()` —
   trafiają do `os.environ`.
5. **Zbuduj connection string** z wczytanych zmiennych (`getenv(...)`) i
   otwórz połączenie przez `mssql_python.connect(...)`.

Przykładowy `.env`:
```
# .env — NIE commitować, dodaj do .gitignore

# Wariant 1: autentykacja SQL (UID/PWD)
SQL_SERVER=localhost,1433
SQL_DATABASE=MojaBaza
SQL_UID=uzytkownik
SQL_PWD=haslo

# Wariant 2: Windows Authentication (np. lokalny SQLEXPRESS)
# — po prostu pomiń SQL_UID/SQL_PWD, kod niżej sam przełączy się
# na Trusted_Connection, gdy nie znajdzie SQL_UID w środowisku
# SQL_SERVER=localhost\SQLEXPRESS
# SQL_DATABASE=Northwind

# tylko na potrzeby lokalnego dev z certyfikatem self-signed:
# SQL_TRUST_SERVER_CERT=yes
```

`mssql_python.connect(...)` przyjmuje **albo** pojedynczy connection string
(jak w kodzie niżej), **albo** parametry nazwane (`server=`, `database=`,
`uid=`, `pwd=`, `trusted_connection=`, `TrustServerCertificate=` — patrz
tabela wyżej) — z punktu widzenia `.env` to bez różnicy, oba warianty
budujesz identycznie przez `getenv(...)`.

`load_dotenv()` domyślnie szuka `.env` w bieżącym katalogu roboczym i w
katalogach nadrzędnych — w notebookach/skryptach odpalanych z różnych
lokalizacji bywa to niejawne i zależne od `cwd`. Bezpieczniej wskazać ścieżkę
wprost: `load_dotenv(Path.cwd() / ".env")` albo stałą ścieżkę do katalogu
repo.

> W pipeline'ach uruchamianych na platformie (Airflow, Azure DevOps itp.)
> `.env` sprawdza się głównie lokalnie/dev. Na produkcji sekrety trzymaj w
> dedykowanym secret storze platformy — w Airflow to **Connections** /
> **Variables** (opcjonalnie z backendem w Key Vault), a nie plik `.env`
> leżący na workerze.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from os import getenv

load_dotenv()  # wczytuje zmienne z .env (bieżący katalog / katalogi nadrzędne) do os.environ

parts = [
    f"Server={getenv('SQL_SERVER')}",
    f"Database={getenv('SQL_DATABASE')}",
    "Encrypt=yes",
]

sql_uid = getenv("SQL_UID")
if sql_uid:                                    # SQL_UID podany w .env -> autentykacja SQL
    parts += [f"UID={sql_uid}", f"PWD={getenv('SQL_PWD')}"]
else:                                          # brak SQL_UID -> Windows Authentication
    parts.append("Trusted_Connection=yes")

if getenv("SQL_TRUST_SERVER_CERT", "no").lower() == "yes":
    parts.append("TrustServerCertificate=yes")  # tylko lokalny dev, certyfikat self-signed

CONNECTION_STRING = ";".join(parts) + ";"

conn = mssql_python.connect(CONNECTION_STRING)

### Ważne właściwości sterownika (inne niż w `pyodbc`)

- **Domyślnie `autocommit=False`** — każda instrukcja żyje w transakcji, dopóki
  nie wywołasz `conn.commit()` (albo `conn.rollback()`). To dobra wiadomość
  dla operacji DML z sekcji 3 — TRUNCATE + INSERT łatwo zamknąć w jedną
  atomową transakcję.
- **Brak MARS** (Multiple Active Result Sets) — na jednym połączeniu może być
  aktywny tylko jeden "niewyczerpany" wynik zapytania na raz. Zanim odpalisz
  kolejne `execute()` na tym samym cursorze (albo drugim cursorze tego
  samego connection), dociągnij wszystkie wiersze poprzedniego zapytania
  (`fetchall()`), albo użyj osobnego połączenia.
- **Context manager** — `with connect(...) as conn:` domyka połączenie na
  wyjściu z bloku (ale NIE robi automatycznego commit/rollback za Ciebie
  poza przypadkiem wyjątku — i tak jawne `conn.commit()` jest najbezpieczniejsze).

In [12]:
# Bezpieczny wzorzec otwierania/zamykania połączenia — poleca się w skryptach/pipeline'ach
with conn:
    cursor = conn.cursor()
    cursor.execute("SELECT 1 AS test")
    print(cursor.fetchone())
    conn.commit()
# połączenie zamknięte automatycznie na wyjściu z bloku `with`

(1)


## 2. Pobranie tabeli / wyniku zapytania SQL do DataFrame

### a) Najprościej — `pandas.read_sql` / `polars.read_database`

Obie biblioteki potrafią wykonać zapytanie bezpośrednio na dowolnym
połączeniu zgodnym z DB-API 2.0 (a `mssql_python` takim jest) — nie trzeba
ręcznie obsługiwać cursora.

In [26]:
query = f"""
SELECT * FROM OrderDetails
"""

df_pandas = pd.read_sql(query, conn)      # najprostszy sposób w Pandas
df_polars = pl.read_database(query, conn) # analogicznie w Polars
#df_duckdb = duckdb.sql(query, conn) # nie zadziała. DuckDB nie ma automatycznego dostępu do bazy - widzi jedynie pliki na dysku oraz zmienne znajdujące się w pamięci Pythona.

df_pandas.head()

C:\Users\pkawk\AppData\Local\Temp\ipykernel_1600\2400603299.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pandas = pd.read_sql(query, conn)      # najprostszy sposób w Pandas


,OrderDetailID,OrderID,ProductID,Quantity
0,1,10248,11,12
1,2,10248,42,10
2,3,10248,72,5
3,4,10249,14,9
4,5,10249,51,40


> Uwaga: `pandas.read_sql` może wypisać `UserWarning` o "tylko SQLAlchemy
> connectable są w pełni wspierane" — to nieszkodliwe; `mssql_python`
> implementuje standardowe DB-API 2.0, więc odczyt działa poprawnie.
> `polars.read_database` z surowym połączeniem DB-API bywa wolniejszy niż
> `read_database_uri` (który korzysta z `connectorx`/`adbc` i pomija Pythona
> przy konwersji do Arrow) — przy naprawdę dużych zapytaniach warto to
> rozważyć, o ile masz gotowy connection URI zamiast connection stringa ODBC.

### b) Zapytania z parametrami — `%s` i dlaczego przecinek w krotce

Wartości do zapytania zawsze przekazuj jako osobny argument
`cursor.execute(sql, params)` — **nigdy** przez `.format()`/f-string
wklejający wartość do tekstu SQL. To otwiera na SQL injection, nawet gdy
wydaje się, że dane są "bezpieczne" (np. pochodzą z Twojego własnego kodu,
a nie od użytkownika — do czasu, aż ktoś to zmieni wyżej w pipeline).

**`%s` to placeholder ("zaślepka")** — miejsce, w które sterownik sam
bezpiecznie wstawi wartość: dokłada cudzysłowy, sprawdza, czy dane nie
zawierają złośliwego kodu.
```sql
WHERE Category = %s
```
Przy kilku parametrach dokładasz kolejne `%s` w kolejności, w jakiej
podajesz wartości:
```sql
WHERE Category = %s AND OrderDate >= %s
```

**Parametry zawsze jako krotka albo lista** — sterownik (jak większość
implementacji DB-API 2.0) oczekuje sekwencji, bo zapytań z wieloma
parametrami może być wiele naraz. Przy **jednym** parametrze rodzi to
typową pułapkę składniową Pythona: sam nawias okrągły **nie** tworzy
krotki — grupuje wyrażenie, tak jak w matematyce `(2 + 2)`. Krotkę robi
przecinek, nie nawiasy.

In [ ]:
# BEZ przecinka -> zwykły string, nie krotka
x = ("Elektronika")
print(type(x))    # <class 'str'>   -> sterownik dostałby tekst zamiast sekwencji, błąd

# Z przecinkiem -> 1-elementowa krotka
y = ("Elektronika",)
print(type(y))    # <class 'tuple'> -> poprawnie

Przy więcej niż jednym parametrze przecinek pojawia się naturalnie
**między** elementami i nie trzeba go dopisywać na końcu —
`params = ("Elektronika", "2024-01-01")` jest już poprawną krotką.
Przecinek na końcu jest potrzebny wyłącznie przy dokładnie jednym
parametrze. Jeśli wolisz o nim nie pamiętać, użyj nawiasu kwadratowego —
lista jednoelementowa (`["Elektronika"]`) działa tak samo dobrze i nie ma
tej niejednoznaczności.

In [ ]:
# jeden parametr -> przecinek w krotce wymagany, albo użyj listy zamiast krotki
cursor.execute(
    "SELECT KlientID, Nazwa FROM Sales.Customers WHERE Miasto = %s",
    ("Warszawa",),          # równoważnie: ["Warszawa"]
)
df_jeden_parametr = pd.DataFrame(
    (tuple(w) for w in cursor.fetchall()),
    columns=[o[0] for o in cursor.description],
)

# kilka parametrów -> naturalna krotka, bez przecinka na końcu
cursor.execute(
    "SELECT KlientID, Nazwa FROM Sales.Customers WHERE Miasto = %s AND Aktywny = %s",
    ("Warszawa", 1),
)

> Alternatywa, która całkiem omija pytanie o przecinek: parametry
> **nazwane** — `%(nazwa)s` + słownik (`{"miasto": "Warszawa"}`), użyte
> już w zapytaniu z pliku `.sql` w sekcji 3 poniżej. Przy jednym parametrze
> bywa czytelniejsze, przy wielu — jawne nazwy w zapytaniu ułatwiają
> code review (widać od razu, który parametr trafia gdzie, bez liczenia
> kolejności `%s`).

### c) Odczyt dużego wyniku partiami (`fetchmany`) zamiast `fetchall`

Przy zapytaniach zwracających miliony wierszy `fetchall()` ładuje wszystko
naraz do pamięci. `fetchmany()` (rozmiar paczki: `cursor.arraysize`) pozwala
przetwarzać wynik strumieniowo — np. dopisując kolejne paczki bezpośrednio
do pliku Parquet zamiast trzymać cały DataFrame w RAM-ie.

In [ ]:
cursor.execute("SELECT * FROM Sales.OrderDetails")
cursor.arraysize = 5000   # rozmiar jednej paczki

wszystkie_ramki = []
while True:
    paczka = cursor.fetchmany()
    if not paczka:
        break
    kolumny = [o[0] for o in cursor.description]
    wszystkie_ramki.append(pl.DataFrame((tuple(w) for w in paczka), schema=kolumny, orient="row"))

df_streamed = pl.concat(wszystkie_ramki) if wszystkie_ramki else pl.DataFrame()

### d) Przeglad metod fetch - ktore i kiedy

| Metoda | Zwraca | Kiedy uzywac |
|---|---|---|
| `fetchone()` | jeden `Row` lub `None` | pojedynczy rekord (np. po `WHERE Id = ...`), sprawdzenie istnienia |
| `fetchval()` | pojedyncza wartosc (1. kolumna 1. wiersza) | `COUNT(*)`, `MAX(...)`, sumy kontrolne - bez rozpakowywania `fetchone()[0]` |
| `fetchmany(size)` | lista `Row` (domyslnie `cursor.arraysize`, **domyslnie = 1**) | strumieniowe przetwarzanie duzych wynikow bez trzymania calosci w RAM |
| `fetchall()` | lista wszystkich pozostalych wierszy | wynik miesci sie wygodnie w pamieci; wygodne do budowy DataFrame jednym strzalem |
| `for row in cursor:` | wiersz po wierszu | pythonic odpowiednik petli po `fetchone()` - cursor jest iterowalny (DB-API 2.0) |

Wazne:
- Kazda z tych metod **konsumuje** wynik zapytania - raz pobranego wiersza nie
  pobierzesz drugi raz; kolejne wywolanie kontynuuje od miejsca, w ktorym
  skonczylo poprzednie.
- `fetchmany()` bez podanego `size` uzywa `cursor.arraysize`, ktore **domyslnie
  wynosi 1** - jesli chcesz paczkowac (jak w przykladzie b) powyzej), zawsze
  ustaw je jawnie, inaczej dostaniesz jeden wiersz na wywolanie.
- `Row` zwracany przez `fetchone()`/`fetchmany()`/`fetchall()`/iteracje wspiera
  dostep po indeksie (`row[0]`), po nazwie kolumny (`row.KolumnaX`, wielkosc
  liter ma znaczenie) i unpacking (`a, b = row`) - wiec budowanie DataFrame
  recznie przez `[o[0] for o in cursor.description]` (jak w sekcjach 2/3) jest
  potrzebne tylko po to, by miec nazwy kolumn; same wartosci mozna tez
  odczytywac przez atrybuty.

In [ ]:
# Ten sam cursor, rozne metody fetch - kazda wymaga osobnego execute(),
# bo wynik poprzedniego zapytania zostal juz skonsumowany

pierwszy = cursor.execute(
    "SELECT TOP 1 * FROM Sales.OrderDetails ORDER BY OrderDetailID"
).fetchone()
print(pierwszy.OrderDetailID, pierwszy.ProductID)   # dostep po nazwie kolumny

liczba = cursor.execute("SELECT COUNT(*) FROM Sales.OrderDetails").fetchval()
print(f"Liczba wierszy: {liczba}")

cursor.execute("SELECT TOP 3 OrderDetailID, ProductID FROM Sales.OrderDetails")
for wiersz in cursor:            # odpowiednik petli z fetchone()
    print(wiersz.OrderDetailID, wiersz.ProductID)

## 3. Pobranie danych na podstawie zapytania zapisanego w pliku `.sql`

Trzymanie zapytań w osobnych plikach `.sql` (zamiast stringów w kodzie
Pythona) daje podświetlanie składni w edytorze, łatwiejsze code review i
możliwość współdzielenia zapytań między Pythonem a np. DAX Studio/SSMS.

In [ ]:
sciezka_sql = Path("queries/klienci_aktywni.sql")

# treść pliku, np.:
# SELECT KlientID, Nazwa, Miasto, DataAktywacji
# FROM Sales.Customers
# WHERE Aktywny = 1

sql_z_pliku = sciezka_sql.read_text(encoding="utf-8")
df = pd.read_sql(sql_z_pliku, conn)

Jeśli zapytanie w pliku ma placeholdery, wykonuj je jako zapytanie
parametryzowane (`cursor.execute(sql, params)`), **nie** przez `.format()`/
f-string wklejający wartości do tekstu SQL — to otwiera na SQL injection,
nawet gdy zapytanie pochodzi "tylko" z Twojego własnego pliku (parametry
mogą pochodzić od użytkownika/formularza wyżej w pipeline).

In [ ]:
# plik queries/klienci_wg_miasta.sql zawiera:
# SELECT KlientID, Nazwa FROM Sales.Customers WHERE Miasto = %(miasto)s

sql_z_parametrem = Path("queries/klienci_wg_miasta.sql").read_text(encoding="utf-8")

cursor.execute(sql_z_parametrem, {"miasto": "Warszawa"})
df_warszawa = pd.DataFrame(
    (tuple(w) for w in cursor.fetchall()),
    columns=[o[0] for o in cursor.description],
)

## 4. Operacje DML — zapis DataFrame do SQL Server

Wspólny punkt wyjścia dla wszystkich przykładów poniżej: DataFrame gotowy do
zapisu (np. po transformacjach w Polars/DuckDB) plus gotowa instrukcja
`INSERT` z placeholderami `%(nazwa_kolumny)s` (domyślny styl `pyformat`
sterownika — nazwy placeholderów muszą odpowiadać kluczom w słowniku).

### Wzorzec produkcyjny dla kazdego zapisu: `try / except` + jawny `rollback()`

Prosty `cursor.executemany(...); conn.commit()` (jak w przykladzie a) ponizej)
wystarcza do jednorazowych, recznych operacji. W pipeline'ie/skrypcie
uruchamianym bez nadzoru **kazda** operacja zmieniajaca dane (INSERT, UPDATE,
TRUNCATE+INSERT, wiele instrukcji w jednej transakcji) powinna byc opakowana
w `try/except` z jawnym `rollback()` w bloku `except`:

```python
try:
    cursor.execute(...)
    cursor.executemany(...)
    conn.commit()
except Exception:
    conn.rollback()
    raise   # nie polykaj wyjatku - pipeline (np. Airflow) musi zobaczyc fail taska
```

Dlaczego to *najbezpieczniejsza* wersja, a nie tylko "dobra praktyka":

1. **Nie polegaj domyslnie na `with conn:` jako gwarancji rollbacku.**
   Zachowanie context managera zmienialo sie miedzy wersjami sterownika -
   starsze wydania `mssql-python` na wyjsciu z bloku `with` tylko **zamykaja**
   polaczenie (bez automatycznego commit/rollback), dopiero nowsze wersje
   (od 1.11.0) robia commit-on-success / rollback-on-exception automatycznie.
   Jawny `try/except/rollback()` dziala identycznie niezaleznie od wersji
   sterownika zainstalowanej na danym srodowisku - nie musisz tego sprawdzac
   ani ufac, ze produkcja i Twoj lokalny venv maja te sama wersje.
2. **Bez `rollback()` nieudana operacja zostawia transakcje otwarta** az do
   zamkniecia polaczenia - a to oznacza wciaz trzymane blokady (np. na
   tabeli po nieudanym `TRUNCATE`), ktore moga blokowac inne procesy, dopoki
   proces Pythona sie nie zakonczy (albo nie zawisnie).
3. **`raise` po `rollback()` jest tak samo wazny jak sam rollback.** Goly
   `except: conn.rollback()` bez `raise` to *cicha* porazka - w Airflow/
   harmonogramie task zglosi sukces, mimo ze nic sie nie zaladowalo.

Ten wzorzec stosujemy dalej konsekwentnie - widac go juz w przykladzie
TRUNCATE + INSERT (podpunkt b), a warto go uzywac tez przy zwyklym
`INSERT`/`UPDATE`, nie tylko przy operacjach wieloetapowych.

In [ ]:
columns = ", ".join(df.columns)
placeholders = ", ".join(["?"]) * len (df.columns)

insert_sql = """
INSERT INTO dbo.[nazwa_tabeli] ({columns})
VALUES ({placeholders})
"""

rekordy = df.to_dicts()          # Polars -> lista dictów
# rekordy = dane.to_pandas().to_dict("records")   # gdybyś startował z Pandas
rekordy[:2]

### a) Insert DataFrame do istniejącej tabeli

`executemany()` używa **wiązania parametrów kolumnami** (column-wise) — to
znacznie szybsze niż wywoływanie `execute()` w pętli po pojedynczych
wierszach, bo sterownik wysyła dane do serwera w większych porcjach zamiast
jednego round-tripu na wiersz.

In [ ]:
try:
    cursor.executemany(insert_sql, rekordy)
    conn.commit()
    print(f"Wstawiono {cursor.rowcount} wierszy")
except Exception:
    conn.rollback()
    raise

### b) TRUNCATE + INSERT (pełne przeładowanie tabeli)

Klasyczny wzorzec "full load" w ETL/ELT — czyścisz tabelę docelową i
ładujesz ją od nowa. `TRUNCATE` jest szybszy niż `DELETE` (nie loguje
usunięcia wiersz po wierszu) i resetuje `IDENTITY`, ale **wymaga uprawnienia
`ALTER` na tabeli i nie zadziała, jeśli inna tabela ma klucz obcy wskazujący
na tę tabelę** — w takim wypadku użyj `DELETE FROM ...` zamiast `TRUNCATE`.

Obie instrukcje muszą być w jednej transakcji: jeśli TRUNCATE się powiedzie,
a INSERT nie, `rollback()` musi cofnąć też TRUNCATE — inaczej zostajesz z
pustą tabelą.

In [ ]:
try:
    cursor.execute("TRUNCATE TABLE dbo.Klienci")

    cursor.executemany(insert_sql, rekordy)
    conn.commit()

    print(f"Przeładowano tabelę — {cursor.rowcount} wierszy")
except Exception:
    conn.rollback()
    raise

### c) Insert tylko rekordów, które jeszcze nie istnieją w tabeli

Dwa podejścia — wybór zależy od wolumenu danych.

**Po stronie Pythona (anti-join w Polars/Pandas)** — proste i wystarczające,
gdy zbiór kluczy istniejących w tabeli mieści się wygodnie w pamięci
(rzędu pojedynczych milionów wierszy):

In [ ]:
istniejace = pl.read_database("SELECT KlientID FROM dbo.Klienci", conn)["KlientID"].to_list()

nowe_rekordy = df.filter(~pl.col("KlientID").is_in(istniejace))

if nowe_rekordy.height > 0:
    try:
        cursor.executemany(insert_sql, nowe_rekordy.to_dicts())
        conn.commit()
    except Exception:
        conn.rollback()
        raise

print(f"Dodano {nowe_rekordy.height} nowych rekordów (pominięto {df.height - nowe_rekordy.height} już istniejących)")

**Po stronie SQL (tabela tymczasowa + `NOT EXISTS`)** — lepsze przy dużych
wolumenach, bo porównanie kluczy robi silnik SQL na serwerze (z indeksem),
zamiast ściągać wszystkie klucze do Pythona. To też baza pod pełny **upsert**
(`MERGE`), gdyby kiedyś potrzebne było też aktualizowanie istniejących
rekordów, a nie tylko dokładanie nowych.

In [ ]:
try:
    cursor.execute("""
    IF OBJECT_ID('tempdb..#StagingKlienci') IS NOT NULL DROP TABLE #StagingKlienci;
    CREATE TABLE #StagingKlienci (KlientID INT, Nazwa NVARCHAR(200), Miasto NVARCHAR(100));
    """)

    cursor.executemany(
        "INSERT INTO #StagingKlienci (KlientID, Nazwa, Miasto) VALUES (%(KlientID)s, %(Nazwa)s, %(Miasto)s)",
        df.to_dicts(),
    )

    cursor.execute("""
    INSERT INTO dbo.Klienci (KlientID, Nazwa, Miasto)
    SELECT s.KlientID, s.Nazwa, s.Miasto
    FROM #StagingKlienci s
    WHERE NOT EXISTS (
        SELECT 1 FROM dbo.Klienci k WHERE k.KlientID = s.KlientID
    );
    """)

    conn.commit()
    print(f"Dodano {cursor.rowcount} nowych rekordów (INSERT ... WHERE NOT EXISTS)")
except Exception:
    conn.rollback()
    raise

> Wariant z pełnym upsertem (`MERGE`), gdyby był potrzebny — po stronie SQL,
> na tej samej tabeli tymczasowej:
> ```sql
> MERGE dbo.Klienci AS target
> USING #StagingKlienci AS source
>     ON target.KlientID = source.KlientID
> WHEN MATCHED THEN
>     UPDATE SET target.Nazwa = source.Nazwa, target.Miasto = source.Miasto
> WHEN NOT MATCHED THEN
>     INSERT (KlientID, Nazwa, Miasto) VALUES (source.KlientID, source.Nazwa, source.Miasto);
> ```

### d) Insert partiami — gdy DataFrame jest duży (10 tys.+ rekordów)

To jedna z niewielu rzeczy w tym notebooku, którą warto mieć jako funkcję —
używasz jej w praktycznie każdym pipeline'ie ładującym więcej niż kilka
tysięcy wierszy, więc pisanie jej na nowo za każdym razem nie ma sensu.

Chunking ma dwa uzasadnienia praktyczne:
1. **Kontrola rozmiaru transakcji** — jeden ogromny `executemany()` na 500
   tys. wierszy trzyma otwartą transakcję bardzo długo (blokady, log
   transakcyjny rośnie); łatwiej commitować co N wierszy.
2. **Widoczność postępu i punkt przywracania** — przy błędzie w połowie
   wiesz, ile już się załadowało, zamiast zaczynać całość od zera.

In [ ]:
total_rows = len(df)
batch_size = 10000  # liczba rekordów do ładowania w jednej partii

with conn.cursor() as cursor:
    cursor.execute("TRUNCATE TABLE dbo.[nazwa_tabeli]")
    conn.commit()

    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        rekordy_partii = df[start:end].to_dicts()   # slicing Polars + konwersja partii na rekordy

        try:
            cursor.executemany(insert_sql, rekordy_partii)
            conn.commit()
        except Exception:
            conn.rollback()   # cofa tylko bieżącą, nie w pełni załadowaną partię
            raise             # wcześniej zacommitowane partie zostają — wiadomo, gdzie wznowić

        print(
            f"Załadowano {end:,} z {total_rows:,} rekordów "
            f"({end/total_rows:.1%})"
        )

print("Ładowanie zakończone")
conn.close()

## 5. Inne przydatne przypadki użycia (analiza + pipeline'y)

### a) UPDATE na podstawie DataFrame (ten sam wzorzec co INSERT)

In [ ]:
zmiany = pl.DataFrame({
    "KlientID": [101, 102],
    "Nazwa": ["Firma A (zmieniona nazwa)", "Firma B Sp. z o.o."],
    "Miasto": ["Warszawa", "Kraków"],
})

update_sql = """
UPDATE dbo.Klienci
SET Nazwa = %(Nazwa)s, Miasto = %(Miasto)s
WHERE KlientID = %(KlientID)s
"""

try:
    cursor.executemany(update_sql, zmiany.to_dicts())
    conn.commit()
    print(f"Zaktualizowano {cursor.rowcount} wierszy")
except Exception:
    conn.rollback()
    raise

### b) DuckDB jako silnik transformacji przed zapisem do SQL Server

DuckDB potrafi odpytywać DataFrame Polars/Pandas bezpośrednio po nazwie
zmiennej (bez kopiowania danych) — wygodne do szybkich agregacji/joinów
tuż przed wysłaniem wyniku z powrotem do SQL Server, szczególnie gdy
transformacja łączy dane z kilku różnych źródeł (np. wynik zapytania SQL +
plik CSV + inny DataFrame).

In [ ]:
podsumowanie = duckdb.sql("""
    SELECT Miasto, COUNT(*) AS liczba_klientow
    FROM dane
    GROUP BY Miasto
    ORDER BY liczba_klientow DESC
""").pl()   # .pl() -> Polars, .df() -> Pandas, .arrow() -> Arrow Table

podsumowanie

### c) Watermark / ładowanie przyrostowe (incremental load)

Zamiast ściągać całą tabelę źródłową za każdym razem — typowy wzorzec w
pipeline'ach (Airflow, harmonogramy nocne): pobierz tylko rekordy zmienione
od ostatniego uruchomienia, zapisując "znak wodny" (np. max. `DataModyfikacji`
poprzedniego przebiegu) w osobnej tabeli kontrolnej.

In [ ]:
ostatni_watermark = cursor.execute(
    "SELECT MAX(WartoscWatermark) FROM etl.Watermarks WHERE NazwaProcesu = %(nazwa)s",
    {"nazwa": "load_klienci"},
).fetchval()

sql_przyrostowy = """
SELECT KlientID, Nazwa, Miasto, DataModyfikacji
FROM Sales.Customers
WHERE DataModyfikacji > %(watermark)s
"""
cursor.execute(sql_przyrostowy, {"watermark": ostatni_watermark})
df_przyrost = pd.DataFrame((tuple(w) for w in cursor.fetchall()), columns=[o[0] for o in cursor.description])

# ... insert_in_batches(...) do tabeli docelowej, a na końcu:
nowy_watermark = df_przyrost["DataModyfikacji"].max()

try:
    cursor.execute(
        "UPDATE etl.Watermarks SET WartoscWatermark = %(w)s WHERE NazwaProcesu = %(nazwa)s",
        {"w": nowy_watermark, "nazwa": "load_klienci"},
    )
    conn.commit()
except Exception:
    conn.rollback()
    raise

### d) Szybkie sprawdzenie pojedynczej wartości — `fetchval()`

Wygodne do kontroli w pipeline (liczba wierszy przed/po, suma kontrolna),
bez rozpakowywania `fetchone()[0]`.

In [ ]:
liczba_wierszy = cursor.execute("SELECT COUNT(*) FROM dbo.Klienci").fetchval()
print(f"Tabela dbo.Klienci: {liczba_wierszy} wierszy")